# claims-support-rl on Colab: 14B GRPO with Claude as judge

Runtime: **A100 80GB** (Runtime > Change runtime type). Secrets: add `ANTHROPIC_API_KEY` under the key icon on the left before running.

Runs 8 and 9 train Qwen2.5-14B-Instruct (bf16, LoRA) with prompt v5 and v8 on the relabeled sub35, judged by `claude-haiku-4-5`. Their rubric scores are not comparable with runs 3-7 (judged by qwen3:30b-a3b on the home box); route accuracy is.

In [ ]:
%pip install -q "torch>=2.4" "transformers==5.17.0" "trl==1.13.0" "peft==0.20.0" accelerate datasets "anthropic==1.4.0"
# Colab ships torchao 0.10, which peft 0.20 rejects on import (needs >=0.16). peft uses it only for
# quantized layers, which this LoRA setup does not touch; peft imports cleanly without it.
%pip uninstall -y -q torchao
import torch, transformers, trl, peft, anthropic
print(torch.__version__, transformers.__version__, trl.__version__, peft.__version__, anthropic.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## Code and data

Everything the notebook needs is in the public repo at commit `cb4130f4f2d3` (reward/, train/, eval/, the KB versions, the relabeled sub35, the stratified test). The next cell clones it and checks that commit out.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q cb4130f4f2d3e24cc9568381670a6e71fa473713 && git log --oneline -1
import json
from collections import Counter
sub = [json.loads(l) for l in open('data/v3_kb_definitions/tasks_train_sub35.jsonl')]
print('sub35 rows', len(sub), '| t110 ->', next(t['route_answer'] for t in sub if t['task_id']=='t110'), '(expect refer_to_manufacturer)')
test = [json.loads(l) for l in open('data/v3_kb_definitions/tasks_test.jsonl')]
print('test routes', dict(sorted(Counter(t['route_answer'] for t in test).items())))

## Judge: Claude through the Messages API

In [ ]:
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['JUDGE_BACKEND'] = 'anthropic'

import sys; sys.path.insert(0, '/content/claims-support-rl')
from reward import judge
print(judge.judge_info())
verdict, reasoning = judge.ask_with_reasoning(
    'Does the reply state the 31-day rule?',
    'Customer message: "I spilled coffee on my laptop 20 days after signing up."\n\n'
    'Support agent reply: "Your plan is active, but repair and replacement coverage starts on day 31 after enrollment."')
print('verdict:', verdict, '| reasoning:', reasoning, '| usage:', judge.LAST)

## Baseline: 14B base, prompt v5, stratified test (judge = anthropic)

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 --score \
    --out "$DRIVE/eval_base14b_v5_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

## run8: 14B, prompt v5, relabeled sub35

run4 settings: 8 generations, batch 2 x accum 4, beta 0, seed 42, 35 steps, default reward weights, LoRA r=16 on the seven projection modules. Checkpoints, completions and live_steps go to Drive. The log prints s/step and peak VRAM after step 2.

In [ ]:
%env JUDGE_BACKEND=anthropic
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 \
    --out "$DRIVE/grpo_run8_14b_v5" --batch 2 --accum 4 --gens 8 --beta 0 --steps 35 --save-steps 35 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

## run8 eval: checkpoint-35, v5, stratified test

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 --score \
    --adapter "$DRIVE/grpo_run8_14b_v5/checkpoint-35" --out "$DRIVE/eval_run8_ckpt35_v5_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
import json
def table(pairs):
    sums = {n: json.load(open(f'{DRIVE}/{d}/summary.json')) for n, d in pairs}
    routes = ['ask_question','file_claim','explain_not_covered','escalate','refer_to_manufacturer','explain_waiting_period','tech_support']
    print('| gold route | ' + ' | '.join(n for n, _ in pairs) + ' |'); print('|---' * (len(pairs) + 1) + '|')
    for r in routes:
        print(f'| {r} | ' + ' | '.join(f"{sums[n]['per_route'][r]['correct']}/{sums[n]['per_route'][r]['total']}" for n, _ in pairs) + ' |')
    print('| **total** | ' + ' | '.join(f"**{round(sums[n]['route_accuracy']*40)}/40**" for n, _ in pairs) + ' |')
    print('| rubric pass rate | ' + ' | '.join(f"{sums[n].get('rubric_pass_rate_mean', float('nan')):.3f}" for n, _ in pairs) + ' |')
    print('judge:', {n: sums[n].get('judge') for n, _ in pairs})
table([('14B base v5', 'eval_base14b_v5_test3'), ('run8 ckpt35 v5', 'eval_run8_ckpt35_v5_test3')])

## run9: same with prompt v8

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --score \
    --out "$DRIVE/eval_base14b_v8_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
%env JUDGE_BACKEND=anthropic
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 \
    --out "$DRIVE/grpo_run9_14b_v8" --batch 2 --accum 4 --gens 8 --beta 0 --steps 35 --save-steps 35 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --score \
    --adapter "$DRIVE/grpo_run9_14b_v8/checkpoint-35" --out "$DRIVE/eval_run9_ckpt35_v8_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'
table([('14B base v8', 'eval_base14b_v8_test3'), ('run9 ckpt35 v8', 'eval_run9_ckpt35_v8_test3')])